# IEEE‑CIS Fraud Detection — High‑Performance Modeling Pipeline

# 1. Import Framework

In [2]:
#load libraries
import pandas as pd
import numpy as np
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier


# 2. Load Data

In [3]:
#Load the Data
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity     = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


# 3. Merge Data

In [4]:
#Merge the Data
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")

# 4. Memory Optimizatio

In [5]:
def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
        else:
            df[col] = df[col].astype("category")
    return df

train = reduce_mem(train)
test  = reduce_mem(test)


# 5. Split Data into Features and Target

In [6]:
#Separate Target and Features
y = train["isFraud"]
X = train.drop("isFraud", axis=1)
X_test = test.copy()


# 6. Align Train And Test Columns

In [7]:
#Align columns between train and test
missing_cols = [col for col in X.columns if col not in X_test.columns]
X_test = pd.concat([X_test, pd.DataFrame(np.nan, index=X_test.index, columns=missing_cols)], axis=1)
X_test = X_test[X.columns]


# 7. Feature Engineering 

In [8]:
# This Collect new features in dictionaries

new = {}
new_test = {}

# --- Time Features ---
new["day"] = X["TransactionDT"] // (24*60*60)
new_test["day"] = X_test["TransactionDT"] // (24*60*60)

new["hour"] = (X["TransactionDT"] // 3600) % 24
new_test["hour"] = (X_test["TransactionDT"] // 3600) % 24

# --- Amount Features ---
new["log_amt"] = np.log1p(X["TransactionAmt"])
new_test["log_amt"] = np.log1p(X_test["TransactionAmt"])

new["amt_per_day"] = X["TransactionAmt"] / (new["day"] + 1)
new_test["amt_per_day"] = X_test["TransactionAmt"] / (new_test["day"] + 1)

# --- Frequency Encoding ---
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]

for col in freq_cols:
    freq = X[col].value_counts()
    new[col + "_freq"] = X[col].map(freq)
    new_test[col + "_freq"] = X_test[col].map(freq)

# --- Device Features ---
new["device_name"] = X["DeviceInfo"].str.split("/", expand=True)[0]
new_test["device_name"] = X_test["DeviceInfo"].str.split("/", expand=True)[0]

new["device_version"] = X["DeviceInfo"].str.split("/", expand=True)[1]
new_test["device_version"] = X_test["DeviceInfo"].str.split("/", expand=True)[1]

# --- Email Domain Grouping ---
email_map = {
    'gmail.com': 'google', 'gmail': 'google',
    'yahoo.com': 'yahoo', 'yahoo.com.mx': 'yahoo',
    'hotmail.com': 'microsoft', 'outlook.com': 'microsoft'
}

new["P_emaildomain_group"] = X["P_emaildomain"].map(email_map)
new_test["P_emaildomain_group"] = X_test["P_emaildomain"].map(email_map)

# this add features at once on Fragmentation


X = pd.concat([X, pd.DataFrame(new)], axis=1)
X_test = pd.concat([X_test, pd.DataFrame(new_test)], axis=1)


# 8. Encode categorical features

In [9]:

categorical_cols = X.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))



# 9. Automatic Column Detection

In [10]:
# Identify numeric and categorical columns:
numeric_cols = X.select_dtypes(include=["int", "float", "float32", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns


# 10. Convert Numeric Columns To Float32

In [11]:
X[numeric_cols] = X[numeric_cols].astype("float32")
X_test[numeric_cols] = X_test[numeric_cols].astype("float32")


# 11. Handle missing values

In [12]:
#Handle Missing Values
num_cols = X.select_dtypes(include=["int", "float"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

X[num_cols] = X[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

X[cat_cols] = X[cat_cols].fillna("missing")
X_test[cat_cols] = X_test[cat_cols].fillna("missing")


# 12.Fill Missing Values

In [13]:
X = X.fillna(-1)
X_test = X_test.fillna(-1)


# 13. Train And Validation Split

In [14]:
#train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# 14. Lightgbm Model

In [15]:
train_set = lgb.Dataset(X_train, y_train)
valid_set = lgb.Dataset(X_valid, y_valid)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 256,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1
}

model_lgb = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=20000,
    callbacks=[lgb.early_stopping(500), lgb.log_evaluation(200)]
)

valid_pred_lgb = model_lgb.predict(X_valid)
auc_lgb = roc_auc_score(y_valid, valid_pred_lgb)
print("LightGBM AUC:", auc_lgb)

Training until validation scores don't improve for 500 rounds
[200]	training's auc: 0.956126	valid_1's auc: 0.901757
[400]	training's auc: 0.983163	valid_1's auc: 0.9206
[600]	training's auc: 0.99323	valid_1's auc: 0.926716
[800]	training's auc: 0.997175	valid_1's auc: 0.929012
[1000]	training's auc: 0.998818	valid_1's auc: 0.929548
[1200]	training's auc: 0.999462	valid_1's auc: 0.929754
[1400]	training's auc: 0.999764	valid_1's auc: 0.930135
[1600]	training's auc: 0.9999	valid_1's auc: 0.929917
[1800]	training's auc: 0.999959	valid_1's auc: 0.930035
Early stopping, best iteration is:
[1466]	training's auc: 0.999822	valid_1's auc: 0.930319
LightGBM AUC: 0.9303192689546266


# 15. CatBoost Block Improved + Tuned

In [16]:
# Detect categorical columns
cat_features = [
    i for i, col in enumerate(X_train.columns)
    if X_train[col].dtype == "category"
]

cat = CatBoostClassifier(
    iterations=4000,
    learning_rate=0.03,
    depth=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_strength=1.5,
    l2_leaf_reg=3,
    border_count=128,
    verbose=False
)

cat.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features
)

valid_pred_cat = cat.predict_proba(X_valid)[:, 1]
auc_cat = roc_auc_score(y_valid, valid_pred_cat)
print("CatBoost AUC:", auc_cat)

CatBoost AUC: 0.9217274536875808


# 16. Xgboost Model

In [17]:

model_xgb = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)


model_xgb.fit(X_train, y_train)

valid_pred_xgb = model_xgb.predict_proba(X_valid)[:, 1]
auc_xgb = roc_auc_score(y_valid, valid_pred_xgb)
print("XGBoost AUC:", auc_xgb)


XGBoost AUC: 0.9271206323754169


# 16 Train The Model

In [18]:
model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=20000,
    callbacks=[
        lgb.early_stopping(500),
        lgb.log_evaluation(200)   
    ]
)


Training until validation scores don't improve for 500 rounds
[200]	training's auc: 0.956126	valid_1's auc: 0.901757
[400]	training's auc: 0.983163	valid_1's auc: 0.9206
[600]	training's auc: 0.99323	valid_1's auc: 0.926716
[800]	training's auc: 0.997175	valid_1's auc: 0.929012
[1000]	training's auc: 0.998818	valid_1's auc: 0.929548
[1200]	training's auc: 0.999462	valid_1's auc: 0.929754
[1400]	training's auc: 0.999764	valid_1's auc: 0.930135
[1600]	training's auc: 0.9999	valid_1's auc: 0.929917
[1800]	training's auc: 0.999959	valid_1's auc: 0.930035
Early stopping, best iteration is:
[1466]	training's auc: 0.999822	valid_1's auc: 0.930319



# 17. Auc Evalution (lgbm)

In [ ]:
cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False
)
cat.fit(X_train, y_train, eval_set=(X_valid, y_valid))
valid_pred_cat = cat.predict_proba(X_valid)[:, 1]
auc_cat = roc_auc_score(y_valid, valid_pred_cat)
print("CatBoost AUC:", auc_cat)

# 18. Xgbboost Model

In [ ]:
cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False
)
cat.fit(X_train, y_train, eval_set=(X_valid, y_valid))
valid_pred_cat = cat.predict_proba(X_valid)[:, 1]
auc_cat = roc_auc_score(y_valid, valid_pred_cat)
print("CatBoost AUC:", auc_cat)

# 19. Stacking (lgbm + catboost + xgboost)

In [ ]:
stack_valid = (
    0.50 * valid_pred_lgb +
    0.30 * valid_pred_cat +
    0.20 * valid_pred_xgb
)

auc_stack = roc_auc_score(y_valid, stack_valid)
print("Stacked AUC:", auc_stack)


# 18. Test Predictions

In [ ]:
test_pred_lgb = model_lgb.predict(X_test)
test_pred_cat = cat.predict_proba(X_test)[:, 1]
test_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

test_pred_stack = (0.5 * test_pred_lgb) + (0.3 * test_pred_cat) + (0.2 * test_pred_xgb)

In [ ]:
test_pred = model.predict(X_test)

# 19. Create Submission File

In [ ]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_pred
})

submission.to_csv("submission.csv", index=False)
